# COMP3132 - Lab Week 1

# Building an LLM-Powered Chatbot: A Hands-On Guide in Google Colab

## Google Colab Configuration

### Some of our labs this semester will be painfully slow if without a GPU. The easies way to get access to a GPU accelerated Jupyter notebook is to enable the `T4 GPU runtime` on Google Colab:

### 1. Navigate to `Runtime`.
### 2. Select `Change runtime type`.
### 3. Choose `Hardware accelerator`.
### 4. Select `T4 GPU`.

### **Note:** This notebook can be run on `CPU` without any noticeable difference in performance.

In [1]:
from IPython.display import Image, display

In [2]:
!pip install python-dotenv
!pip install jupyter_bokeh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


# Online Chatbot

### Go to https://api.together.ai/playground/chat/meta-llama/Llama-2-7b-chat-hf to chat with the model online on `togerther.ai` website and play with the chatbot by changing the configurations and hyper-parameters

# A Brief Theory




## Training a Language Model

In [3]:
image_path = './assets/LLM_train.png'
display(Image(image_path, width=600))

FileNotFoundError: No such file or directory: './assets/LLM_train.png'

FileNotFoundError: No such file or directory: './assets/LLM_train.png'

<IPython.core.display.Image object>

## Base Vs. Chat Models

### After training the LLMs with this paradigm on a very large amount of data (such as the entire internet), we will have a model, also known as a `foundation` model or `base` model, that can predict the next word repeatedly to form a sentence.

### To enable the model to engage in conversations, we further fine-tune the base model using instructions, such as question-answer pairs. These models are referred to as `instruction-tuned` or `chat` models.

### You can observe the different behaviors of the base and instruction-tuned models in the following slide.

In [4]:
image_path = './assets/baseVSinstruct.png'
display(Image(image_path, width=800))


FileNotFoundError: No such file or directory: './assets/baseVSinstruct.png'

FileNotFoundError: No such file or directory: './assets/baseVSinstruct.png'

<IPython.core.display.Image object>

## Interacting with Model Programmatically

In [5]:
image_path = './assets/modelaccess.png'
display(Image(image_path, width=500))

FileNotFoundError: No such file or directory: './assets/modelaccess.png'

FileNotFoundError: No such file or directory: './assets/modelaccess.png'

<IPython.core.display.Image object>

# Designing Our Own Chatbot

## API Call to the Model

### Getting API KEY

#### - Go to https://api.together.xyz/settings/api-keys to get your API key.

#### Importing the API Key to Colab

1. On the left-side vertical menu, select the `key` icon.
2. Add a secret key with the following details:
   - **Name**: `TOGETHER_API_KEY`
   - **Value**: `<your API key>`

In [6]:
from google.colab import userdata
api_key = userdata.get('TOGETHER_API_KEY')

### Function to call the API

In [7]:
import os
# from dotenv import load_dotenv, find_dotenv
import warnings
import requests
import json
import time

warnings.filterwarnings('ignore')
url = "https://api.together.xyz/inference"

headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }


import time
def llama(prompt,
          add_inst=True,
          model="meta-llama/Llama-2-7b-chat-hf",
          temperature=0.0,
          max_tokens=1024,
          verbose=False,
          url=url,
          headers=headers,
          base = 2, # number of seconds to wait
          max_tries=3):

    if add_inst:
        prompt = f"[INST]{prompt}[/INST]"

    if verbose:
        print(f"Prompt:\n{prompt}\n")
        print(f"model: {model}")

    data = {
            "model": model,
            "prompt": prompt,
            "temperature": temperature,
            "max_tokens": max_tokens
        }

    # Allow multiple attempts to call the API incase of downtime.
    # Return provided response to user after 3 failed attempts.
    wait_seconds = [base**i for i in range(max_tries)]

    for num_tries in range(max_tries):
        try:
            response = requests.post(url, headers=headers, json=data)
            return response.json()['output']['choices'][0]['text']
        except Exception as e:
            if response.status_code != 500:
                return response.json()

            print(f"error message: {e}")
            print(f"response object: {response}")
            print(f"num_tries {num_tries}")
            print(f"Waiting {wait_seconds[num_tries]} seconds before automatically trying again.")
            time.sleep(wait_seconds[num_tries])

    print(f"Tried {max_tries} times to make API call to get a valid response object")
    print("Returning provided response")
    return response


### **Note:** Default model is `"meta-llama/Llama-2-7b-chat-hf"` but can you can change it by finding the model name from https://api.together.ai/playground/chat

## General testing the model

In [8]:
# pass prompt to the llama function, store output as 'response' then print
prompt = "Tell me a joke about software developers."
response = llama(prompt, temperature=0.0)  # temperature is a hyperparameter that controls randomness in the response
print(response)

  Sure, here's a classic one:

Why do software developers prefer dark mode?

Because light attracts bugs.

(Note: This joke is a play on words, as "light" can refer to both the color and the source of illumination, and "bugs" is a common term used in software development to refer to errors or problems.)


In [9]:
prompt = "What is the capital of France?"
response = llama(prompt, verbose=True) # verbose=True will print the prompt
print(response)

Prompt:
[INST]What is the capital of France?[/INST]

model: meta-llama/Llama-2-7b-chat-hf
  The capital of France is Paris.


## Exercise 1: General testing

#### 1. Change the `temprarature` parameter from 0.0 to 0.9 and see the difference in the responses.
#### Note: temperature parameter is a number between 0.0 and 1.0. It controls the randomness of the responses.

In [10]:
#your code here

### 2. what does `verbose` argument do?

#### your answer here

## Role prompting

#### - Roles give context to LLMs what type of answers are desired.
#### - LLMs often gives more consistent responses when provided with a role.
#### - First, try standard prompt and see the response.

In [11]:
prompt = """
How can I answer this question from my friend:
What is the meaning of life?
"""
response = llama(prompt)
print(response)

{'id': '901231ce1b12ebc1-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}


###  Now, try it by giving the model a `role`, and within the role, a `tone` using which it should respond with.

In [12]:
role = """
Your role is a life coach \
who gives advice to people about living a good life.\
You attempt to provide unbiased advice.
You respond in the tone of an English pirate.
"""

prompt = f"""
{role}
How can I answer this question from my friend:
What is the meaning of life?
"""
response = llama(prompt)
print(response)

{'id': '901231cead75a374-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}


## Excercise 2: Role prompting

#### Role: Beginner python tutor
#### Task: Explain how to create a list and add an element to it.

In [13]:
# your code here

#### Change the role to `friendly coding mentor` and see how the response changes for the same task.

In [14]:
# your code here

## Asking follow-up questions

### Does the model have memory of the previous conversation?

In [15]:
prompt_1 = """
What are fun activities I can do this weekend?
"""

response_1 = llama(prompt_1)
print(response_1)

{'id': '901231cf5a5bd44a-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}


In [16]:
prompt_2 = """
Which of these would be good for my health?
"""
response_2 = llama(prompt_2)
print(response_2)

{'id': '901231cfdb20ba3a-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}


#### Is the the second answer related to the first answer?
#### **Note:** LLMs are `stateless` models, so they don't have memory of the previous conversation.

## Multi-turn prompting (chatting)
#### In order to give the model memory of the previous conversation, you need to provide prior prompts and responses as part of the context of each new turn in the conversation.

In [17]:
image_path = './assets/multi_turn.png'
display(Image(image_path, width=600))

FileNotFoundError: No such file or directory: './assets/multi_turn.png'

FileNotFoundError: No such file or directory: './assets/multi_turn.png'

<IPython.core.display.Image object>

### Note: you donit need `end tag (</s>)` for the last prompt.

In [18]:
chat_prompt = f"""
<s>[INST] {prompt_1} [/INST]
{response_1}
</s>
<s>[INST] {prompt_2} [/INST]
"""
print(chat_prompt)


<s>[INST] 
What are fun activities I can do this weekend?
 [/INST]
{'id': '901231cf5a5bd44a-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}
</s>
<s>[INST] 
Which of these would be good for my health?
 [/INST]



### Note: pay attention to add_inst (add instruction) argument below

In [19]:
response_2 = llama(chat_prompt,
                 add_inst=False,
                 verbose=True)

Prompt:

<s>[INST] 
What are fun activities I can do this weekend?
 [/INST]
{'id': '901231cf5a5bd44a-SEA', 'error': {'message': 'Request was rejected due to request rate limiting. Your rate limits are 60 RPM (1 QPS) and 60000 TPM (1000 TPS). See details: https://docs.together.ai/docs/rate-limits', 'type': 'rate_limit', 'param': None, 'code': None}}
</s>
<s>[INST] 
Which of these would be good for my health?
 [/INST]


model: meta-llama/Llama-2-7b-chat-hf


In [20]:
print(response_2)

I'm just an AI, I don't have access to personal information or medical history, so I cannot provide personalized health advice. However, I can suggest some general activities that can be beneficial for overall health and well-being:

1. Exercise: Regular physical activity can help improve cardiovascular health, reduce stress, and boost mood. Find an activity you enjoy, such as walking, running, swimming, or dancing, and aim to do it for at least 30 minutes a day.
2. Meditation and mindfulness: Mindfulness practices, such as meditation, deep breathing, or yoga, can help reduce stress and anxiety, improve sleep quality, and boost mood. Aim to set aside 10-15 minutes each day for mindfulness practice.
3. Connect with nature: Spending time in nature can have a calming effect and improve mood. Try to spend at least 30 minutes outdoors each day, whether it's walking in a park, gardening, or simply sitting in a sunny spot.
4. Socialize: Social connections are important for overall health and 

### Helper function to handle multi-turn prompting

### **Note:** You don’t need to understand every part of the helper function. In the next section, you’ll see how to use it in your code.

In [21]:
def llama_chat(prompts,
               responses,
               model="meta-llama/Llama-2-7b-chat-hf",
               temperature=0.0,
               max_tokens=1024,
               verbose=False,
               url=url,
               headers=headers,
               base=2,
               max_tries=3
              ):

    prompt = get_prompt_chat(prompts,responses)

    # Allow multiple attempts to call the API incase of downtime.
    # Return provided response to user after 3 failed attempts.
    wait_seconds = [base**i for i in range(max_tries)]

    for num_tries in range(max_tries):
        try:
            response = llama(prompt=prompt,
                             add_inst=False,
                             model=model,
                             temperature=temperature,
                             max_tokens=max_tokens,
                             verbose=verbose,
                             url=url,
                             headers=headers
                            )
            return response
        except Exception as e:
            if response.status_code != 500:
                return response.json()

            print(f"error message: {e}")
            print(f"response object: {response}")
            print(f"num_tries {num_tries}")
            print(f"Waiting {wait_seconds[num_tries]} seconds before automatically trying again.")
            time.sleep(wait_seconds[num_tries])

    print(f"Tried {max_tries} times to make API call to get a valid response object")
    print("Returning provided response")
    return response


def get_prompt_chat(prompts, responses):
  prompt_chat = f"<s>[INST] {prompts[0]} [/INST]"
  for n, response in enumerate(responses):
    prompt = prompts[n + 1]
    prompt_chat += f"\n{response}\n </s><s>[INST] \n{ prompt }\n [/INST]"

  return prompt_chat

### How to use the helper function

In [22]:
prompt_1 = """
    What are fun activities I can do this weekend?
"""

response_1 = llama(prompt_1)

In [23]:
prompt_2 = """
Which of these would be good for my health?
"""

In [24]:
prompts = [prompt_1,prompt_2]
responses = [response_1]

In [25]:
prompts = [prompt_1,prompt_2]
responses = [response_1]

# Pass prompts and responses to llama_chat function
response_2 = llama_chat(prompts,responses,verbose=True)

Prompt:
<s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]
  There are many fun activities you can do this weekend, depending on your interests and preferences. Here are some ideas:

1. Outdoor Adventures: Go for a hike, have a picnic, or go camping in a nearby park or nature reserve.
2. Cultural Events: Attend a concert, play, or festival in your area. Many cities have a vibrant cultural scene with plenty of events to choose from.
3. Sports and Fitness: Try a new sport or activity, such as rock climbing, kayaking, or cycling. Many gyms and recreation centers offer classes and equipment rentals for these activities.
4. Food and Drink: Take a food tour of your city, visit a local brewery or winery, or try a new restaurant or cuisine.
5. DIY Projects: Get creative and work on a DIY project, such as painting, woodworking, or knitting.
6. Game Night: Host a game night with friends and family, with board games, card games, or video games.
7. Movie Night: Have a movie mar

In [26]:
print(response_2)

  It's great that you're thinking about your health! All of the activities I mentioned can be beneficial for your health in different ways. Here are some specific health benefits associated with each activity:

1. Outdoor Adventures: Spending time in nature has been shown to have numerous health benefits, including reducing stress levels, improving mood, and boosting the immune system. Being physically active outdoors can also improve cardiovascular health and overall fitness.
2. Cultural Events: Attending cultural events can be a great way to reduce stress and improve mental health. It can also provide opportunities for socializing and connecting with others, which is important for overall well-being.
3. Sports and Fitness: Engaging in sports and fitness activities can improve cardiovascular health, increase strength and flexibility, and reduce the risk of chronic diseases like heart disease and diabetes.
4. Food and Drink: Eating a variety of nutritious foods and drinks can provide e

### Excercise 3: Multi-turn prompting

### Ask this follow-up question: "Which of these activites would be fun with friends?"

In [27]:
prompt_3 = """
Which of these activites would be fun with friends?
"""

chat_prompt = f"""
<s>[INST] {prompt_1} [/INST]
{response_1}
</s>
<s>[INST] {prompt_2} [/INST]
{response_2}
</s>
<s> [INST] {prompt_3} [/INST]
"""
response_3 = llama(chat_prompt, add_inst=False, verbose = True)
print(response_3)
#your code here

Prompt:

<s>[INST] 
    What are fun activities I can do this weekend?
 [/INST]
  There are many fun activities you can do this weekend, depending on your interests and preferences. Here are some ideas:

1. Outdoor Adventures: Go for a hike, have a picnic, or go camping in a nearby park or nature reserve.
2. Cultural Events: Attend a concert, play, or festival in your area. Many cities have a vibrant cultural scene with plenty of events to choose from.
3. Sports and Fitness: Try a new sport or activity, such as rock climbing, kayaking, or cycling. Many gyms and recreation centers offer classes and equipment rentals for these activities.
4. Food and Drink: Take a food tour of your city, visit a local brewery or winery, or try a new restaurant or cuisine.
5. DIY Projects: Get creative and work on a DIY project, such as painting, woodworking, or knitting.
6. Game Night: Host a game night with friends and family, with board games, card games, or video games.
7. Movie Night: Have a movie ma

### OrderBot

#### We can `automate the collection of user prompts and model responses` to build a  OrderBot.

#### The OrderBot will take orders at a pizza restaurant.

In [28]:
# Define the bot's role and menu
role = """
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then start collecting the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \

Primary Category: Pizza
  Secondary Category: \
    pepperoni pizza  12.95, 10.00, 7.00 \
    cheese pizza   10.95, 9.25, 6.50 \
    eggplant pizza   11.95, 9.75, 6.75 \
Primary Category: Sides
  Secondary Category: \
    fries 4.50, 3.50 \
    greek salad 7.25 \
Primary Category: Toppings: \
  Secondary Category: \
    extra cheese 2.00, \
    mushrooms 1.50 \
    sausage 3.00 \
    canadian bacon 3.50 \
    AI sauce 1.50 \
    peppers 1.00 \
Primary Category: Drinks \
  Secondary Category: \
    coke 3.00, 2.00, 1.00 \
    sprite 3.00, 2.00, 1.00 \
    bottled water 5.00 \

the price based on size example:
pepperoni pizza  large = 12.95,
pepperoni pizza  medium = 10.00,
pepperoni pizza  small = 7.00 \

For all items also check the size with the customer first
Do not forget to ask for drinks and sides.
Do not add any items extra by yourself
"""
 # accumulate messages

## Excercise 4: Orderbot

In [ ]:
prompts = []
responses = []

prompts.append(role)
response = llama_chat(prompts, responses)
responses.append(response)
print(f"\nChatbot: {response}\n")

while True:
    # your code here

    # get the input from the user with input() function
    # change prompts and responses accordingly
    # terminate the loop if the user types 'done', 'exit', 'quit', or bye
    ...


Chatbot:   Hello there! Welcome to our pizza joint! 🍕👋 I'm OrderBot, your personal assistant for placing your order. What can I get for you today? 🤔

We have a variety of options in our menu, including pizzas, sides, and drinks. Our pizza categories are:

* Primary Category: Pizza
	+ Secondary Category: Pepperoni Pizza ($12.95, $10.00, $7.00)
		- Large: $12.95
		- Medium: $10.00
		- Small: $7.00
	+ Secondary Category: Cheese Pizza ($10.95, $9.25, $6.50)
		- Large: $10.95
		- Medium: $9.25
		- Small: $6.50
	+ Secondary Category: Eggplant Pizza ($11.95, $9.75, $6.75)
		- Large: $11.95
		- Medium: $9.75
		- Small: $6.75
* Primary Category: Sides
	+ Secondary Category: Fries ($4.50, $3.50)
		- Large: $4.50
		- Medium: $3.50
	+ Secondary Category: Greek Salad ($7.25)
* Primary Category: Toppings
	+ Secondary Category: Extra Cheese ($2.00)
		- Large: $2.00
		- Medium: $2.00
		- Small: $2.00
	+ Secondary Category: Mushrooms ($1.50)
		- Large: $1.50
		- Medium: $1.50
		- Small: $1.50
	+ Secon

### Printing the order

In [ ]:
role = 'create a json summary of the food order. Itemize the price for each item\
 The fields should be 1) pizza, include type of pizza and size and price 2) list of toppings with price 3) list of drinks, include size and price\
          4) list of sides include size and price 5)total price - just include items in my order and do not add anything by yourself'

messages = prompts.copy()
messages.append(role)

response = llama_chat(messages, responses)
print(response)

## Excercise 5: Improving the OrderBot

### Try to improve the performance of your chat by

### 1. modifying the prompt
### 2. using different models (larger ones)